Brochure Generator - Building a website scraping program that utilizes the power of Frontier API's specifically
OpenAI and Anthropic models. The use of Gradio enables a UI interface that we can test our streaming models to 
produce a simple but effective brochure for a chosen company through its website.

In [1]:
# load imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyCO


Calling our models from OpenAI. 

In [3]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

Using "scraper", this allows the models to call the function "fetch_website_contents" to retireve
contents to read through and produce a responsive action based on our system message prompt.


In [4]:
from scraper import fetch_website_contents

In [21]:
system_message = """
    You are a helpful assistant that will help create a brochure from a company website.
    You will be given a website and you will need to create a brochure from the website.
    The brochure will be for prospective clients, investors and recruits.
    Respond in Markdown without code blocks.
    """

In [27]:
def stream_gpt(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result  


In [37]:
def stream_claude(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    stream = anthropic.chat.completions.create(
        model="claude-sonnet-4-5-20250929",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [35]:
def stream_brochure(company,website, model):
    prompt = f"Please generate a brochure for the following company: {company} from the following website: {website}"
    prompt += fetch_website_contents(website)
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Claude":
        result = stream_claude(prompt)
    else:
        raise ValueError(f"Invalid model: {model}")
    yield from result




In [38]:
company = gr.Textbox(label="Company Name:")
website = gr.Textbox(label="Website URL:")
model = gr.Radio(choices=["GPT", "Claude"], value="GPT", label="Model:")
message_output = gr.Markdown(label="Brochure:")

view = gr.Interface(
    fn=stream_brochure,
    inputs=[company, website, model],
    outputs=[message_output],
    title="Brochure Generator",
    examples = [["Constructor","https://constructor.com/","GPT"],["ARC", "https://www2.arccorp.com/","Claude"]],
    flagging_mode="never",
)
view.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
